In [18]:
import ctypes
import time
import random
import cv2
import numpy as np
import pyautogui as pyg

try:
    import mss  # 可選：pip install mss 後截圖速度快很多
except ImportError:
    mss = None

# ============================================================
# 設定區（遇到問題時，調整這裡就好）
# ============================================================
GAME_DURATION = 59.5      # 遊戲時間（秒）
MOVE_MODE = "drag"        # "drag" 拖曳 / "click" 點選兩下（拖曳不靈時改成 click 試試）

# --- 辨識 ---
SAMPLE_FRAC = 0.22        # 每格中心取樣範圍佔格寬比例（越小越只抓寶石內核）
COLOR_THRESH = 18.0       # 同色被拆成多種 → 調大；不同色被合併 → 調小
L_WEIGHT = 0.5            # 亮度權重（降低高光/陰影影響）

# --- 等待盤面穩定（準確度的關鍵）---
SWAP_ANIM_WAIT = 0.20     # 交換後固定先等這麼久（讓交換動畫跑起來）
STABLE_CELL_DIFF = 6.0    # 某格顏色變化超過此值 → 視為「有東西在動」
STABLE_MAX_CHANGED = 1    # 允許幾格在動（容忍閃光特效）
STABLE_FRAMES = 2         # 連續幾次靜止才算穩定
STABLE_TIMEOUT = 0.9      # 等待穩定的最長秒數（超時就直接用當前畫面）

# --- 失敗判斷 ---
MIN_CHANGED_OK = 3        # 交換後至少幾格有變化才算成功消除
FAIL_COOLDOWN = 1.5       # 失敗的走法暫時禁用秒數
DEBUG = True              # 印出辨識結果、存 debug 圖
MAX_FAIL_IMAGES = 5       # 最多存幾張失敗截圖（fail_0.png ...）

pyg.PAUSE = 0
pyg.FAILSAFE = True       # 滑鼠移到螢幕左上角可緊急中止

try:
    ctypes.windll.shcore.SetProcessDpiAwareness(2)
except Exception:
    pass


# ============================================================
# 校準：點兩顆寶石的中心（按空白鍵記錄）
# ============================================================
def wait_space():
    """等待按下空白鍵（Windows 免安裝套件；否則試 keyboard 套件；都沒有就倒數 5 秒）"""
    try:
        user32 = ctypes.windll.user32
        VK_SPACE = 0x20
        while user32.GetAsyncKeyState(VK_SPACE) & 0x8000:        # 先等放開
            time.sleep(0.01)
        while not (user32.GetAsyncKeyState(VK_SPACE) & 0x8000):  # 等按下
            time.sleep(0.01)
        while user32.GetAsyncKeyState(VK_SPACE) & 0x8000:        # 等放開，避免連按
            time.sleep(0.01)
        return
    except Exception:
        pass
    try:
        import keyboard
        keyboard.wait("space")
        time.sleep(0.15)
        return
    except Exception:
        print("  （無法偵測空白鍵，改為倒數 5 秒）")
        time.sleep(5)


def pick_point(msg):
    print(msg + "，然後按【空白鍵】記錄")
    wait_space()
    p = pyg.position()
    print(f"  ✅ 已記錄：({p.x}, {p.y})")
    return p


print("=" * 50)
print("【Bejeweled Blitz - 優化版 v3.1】")
print("=" * 50)
while True:
    p1 = pick_point("👉 把滑鼠移到【左上角第一顆】寶石正中央")
    p2 = pick_point("👉 把滑鼠移到【右下角最後一顆】寶石正中央")
    cell_x = (p2.x - p1.x) / 7.0   # 兩點相隔 7 格
    cell_y = (p2.y - p1.y) / 7.0
    if cell_x > 20 and cell_y > 20 and abs(cell_x - cell_y) <= 1.0:
        break
    print(f"⚠️ 定位不準：橫向格距 {cell_x:.1f}px、縱向 {cell_y:.1f}px 不一致"
          f"（棋盤格是正方形，兩者應該幾乎相同）。請重新定位兩顆寶石。")

CELL = (cell_x + cell_y) / 2.0
LEFT = int(round(p1.x - CELL / 2))
TOP = int(round(p1.y - CELL / 2))
SIZE = int(round(CELL * 8))
print(f"格距 = {CELL:.2f}px，棋盤區域 = ({LEFT}, {TOP}, {SIZE}x{SIZE})")


def gem_center(r, c):
    """第 r 列、第 c 行寶石的螢幕中心座標"""
    return int(p1.x + c * CELL), int(p1.y + r * CELL)


# ============================================================
# 只截棋盤區域
# ============================================================
_sct = mss.mss() if mss else None
_monitor = {"left": LEFT, "top": TOP, "width": SIZE, "height": SIZE}


def grab_board():
    if _sct is not None:
        return np.ascontiguousarray(np.asarray(_sct.grab(_monitor))[:, :, :3])  # BGR
    shot = pyg.screenshot(region=(LEFT, TOP, SIZE, SIZE))
    return cv2.cvtColor(np.array(shot), cv2.COLOR_RGB2BGR)


# ============================================================
# 辨識：每格核心取中位數 → 自動分群成寶石編號
# ============================================================
def sample_cells(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    cs = img_bgr.shape[0] / 8.0
    rad = max(1, int(cs * SAMPLE_FRAC))
    feats = np.zeros((64, 3), np.float32)
    for r in range(8):
        for c in range(8):
            cy, cx = int((r + 0.5) * cs), int((c + 0.5) * cs)
            patch = lab[max(0, cy - rad):cy + rad + 1, max(0, cx - rad):cx + rad + 1]
            feats[r * 8 + c] = np.median(patch.reshape(-1, 3), axis=0)
    feats[:, 0] *= L_WEIGHT
    return feats


def classify(feats):
    """貪婪分群 + 兩輪重新指派，回傳 8x8 的整數寶石編號"""
    centers = []
    labels = np.zeros(64, dtype=int)
    for i in range(64):
        best, bd = -1, 1e9
        for k, ctr in enumerate(centers):
            d = np.linalg.norm(feats[i] - ctr)
            if d < bd:
                best, bd = k, d
        if best >= 0 and bd < COLOR_THRESH:
            labels[i] = best
        else:
            centers.append(feats[i].copy())
            labels[i] = len(centers) - 1

    for _ in range(2):
        C = np.array([feats[labels == k].mean(axis=0) if (labels == k).any() else centers[k]
                      for k in range(len(centers))])
        dist = np.linalg.norm(feats[:, None, :] - C[None, :, :], axis=2)
        labels = dist.argmin(axis=1)
        centers = list(C)

    return labels.reshape(8, 8).tolist()


# ============================================================
# 找走法：只檢查被交換兩格附近的連線
# ============================================================
LEN_SCORE = {3: 10, 4: 30, 5: 60}


def run_len(g, r, c, dr, dc):
    v = g[r][c]
    n = 1
    for s in (1, -1):
        rr, cc = r + dr * s, c + dc * s
        while 0 <= rr < 8 and 0 <= cc < 8 and g[rr][cc] == v:
            n += 1
            rr += dr * s
            cc += dc * s
    return n


def cell_score(g, r, c):
    h = run_len(g, r, c, 0, 1)
    v = run_len(g, r, c, 1, 0)
    s = 0
    if h >= 3:
        s += LEN_SCORE[min(h, 5)]
    if v >= 3:
        s += LEN_SCORE[min(v, 5)]
    if h >= 3 and v >= 3:
        s += 20  # L / T 型加分
    return s


def find_moves(g):
    moves = []
    for r in range(8):
        for c in range(8):
            for dr, dc in ((0, 1), (1, 0)):
                tr, tc = r + dr, c + dc
                if tr >= 8 or tc >= 8 or g[r][c] == g[tr][tc]:
                    continue
                g[r][c], g[tr][tc] = g[tr][tc], g[r][c]
                s = cell_score(g, r, c) + cell_score(g, tr, tc)
                g[r][c], g[tr][tc] = g[tr][tc], g[r][c]
                if s > 0:
                    moves.append((s, r, c, tr, tc))
    return moves


# ============================================================
# 操作
# ============================================================
def do_move(x1, y1, x2, y2):
    if MOVE_MODE == "click":
        pyg.click(x1, y1)
        time.sleep(0.03)
        pyg.click(x2, y2)
    else:
        pyg.moveTo(x1, y1)
        pyg.mouseDown(button="left")
        time.sleep(0.01)
        pyg.moveTo((x1 + x2) // 2, (y1 + y2) // 2)
        time.sleep(0.01)
        pyg.moveTo(x2, y2)
        time.sleep(0.01)
        pyg.mouseUp(button="left")


def changed_cells(f1, f2):
    """兩次取樣之間，有幾格的顏色明顯不同"""
    return int((np.linalg.norm(f1 - f2, axis=1) > STABLE_CELL_DIFF).sum())


def stable_board():
    """
    等到每一格的顏色連續 STABLE_FRAMES 次都沒變（寶石不再掉落/爆炸），
    回傳 (畫面, 特徵, 是否真的穩定)。每次決策都只用穩定的盤面，這是準確度的關鍵。
    """
    t0 = time.time()
    frame = grab_board()
    feats = sample_cells(frame)
    ok = 0
    while True:
        time.sleep(0.03)
        frame2 = grab_board()
        feats2 = sample_cells(frame2)
        n = changed_cells(feats, feats2)
        frame, feats = frame2, feats2
        if n <= STABLE_MAX_CHANGED:
            ok += 1
            if ok >= STABLE_FRAMES:
                return frame, feats, True
        else:
            ok = 0
        if time.time() - t0 > STABLE_TIMEOUT:
            return frame, feats, False


def save_debug(frame, grid, move, name):
    """存圖：每格標上辨識編號，紅色箭頭是程式想做的交換"""
    img = frame.copy()
    cs = img.shape[0] / 8.0
    for r in range(8):
        for c in range(8):
            pos = (int(c * cs + cs * 0.35), int(r * cs + cs * 0.62))
            cv2.putText(img, str(grid[r][c]), pos, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
            cv2.putText(img, str(grid[r][c]), pos, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    _, r1, c1, r2, c2 = move
    a = (int((c1 + 0.5) * cs), int((r1 + 0.5) * cs))
    b = (int((c2 + 0.5) * cs), int((r2 + 0.5) * cs))
    cv2.arrowedLine(img, a, b, (0, 0, 255), 3)
    cv2.imwrite(name, img)


# ============================================================
# 主迴圈
# ============================================================
print("\n🚀 3 秒後開始，請放開滑鼠！（緊急中止：滑鼠甩到螢幕左上角）")
time.sleep(3)

start = time.time()
cooldown = {}   # (r, c, tr, tc) -> 解禁時間
stats = {"moves": 0, "ok": 0, "fail": 0, "unstable": 0}
fail_streak = 0
warned = False

frame, feats, _ = stable_board()
first = True

while time.time() - start < GAME_DURATION:
    now = time.time()
    grid = classify(feats)

    if first and DEBUG:
        first = False
        print("辨識結果（數字 = 寶石種類，通常應為 6~7 種）：")
        for row in grid:
            print("  " + " ".join(str(v) for v in row))
        print(f"共 {len(set(v for row in grid for v in row))} 種")
        cv2.imwrite("debug_board.png", frame)

    moves = [m for m in find_moves(grid)
             if cooldown.get((m[1], m[2], m[3], m[4]), 0) <= now]

    if not moves:
        frame, feats, _ = stable_board()
        continue

    best = max(m[0] for m in moves)
    move = random.choice([m for m in moves if m[0] == best])
    _, r1, c1, r2, c2 = move

    pre_frame, pre_feats, pre_grid = frame, feats, grid
    x1, y1 = gem_center(r1, c1)
    x2, y2 = gem_center(r2, c2)
    do_move(x1, y1, x2, y2)
    stats["moves"] += 1

    time.sleep(SWAP_ANIM_WAIT)
    frame, feats, stable = stable_board()
    if not stable:
        stats["unstable"] += 1

    if changed_cells(pre_feats, feats) >= MIN_CHANGED_OK:
        stats["ok"] += 1
        fail_streak = 0
    else:
        # 盤面沒變 → 這步沒生效（沒收到操作，或辨識錯誤被遊戲退回）
        stats["fail"] += 1
        fail_streak += 1
        cooldown[(r1, c1, r2, c2)] = time.time() + FAIL_COOLDOWN
        if DEBUG and stats["fail"] <= MAX_FAIL_IMAGES:
            save_debug(pre_frame, pre_grid, move, f"fail_{stats['fail'] - 1}.png")
        if fail_streak >= 5 and not warned:
            warned = True
            print("⚠️ 連續失敗 5 次以上：請看 fail_*.png，確認數字是否和實際寶石一致，"
                  "或把 MOVE_MODE 改成 \"click\" 試試。")

total = max(1, stats["moves"])
print(f"時間到！共 {stats['moves']} 次交換：成功 {stats['ok']}、失敗 {stats['fail']}"
      f"（成功率 {stats['ok'] * 100 // total}%），等待穩定超時 {stats['unstable']} 次。")    

【Bejeweled Blitz - 優化版 v3.1】
👉 把滑鼠移到【左上角第一顆】寶石正中央，然後按【空白鍵】記錄
  ✅ 已記錄：(1355, 492)
👉 把滑鼠移到【右下角最後一顆】寶石正中央，然後按【空白鍵】記錄
  ✅ 已記錄：(1695, 838)
格距 = 49.00px，棋盤區域 = (1330, 468, 392x392)

🚀 3 秒後開始，請放開滑鼠！（緊急中止：滑鼠甩到螢幕左上角）
辨識結果（數字 = 寶石種類，通常應為 6~7 種）：
  0 1 2 0 3 4 1 5
  3 6 7 0 6 7 6 4
  7 4 0 2 3 4 7 8
  4 7 2 8 4 8 8 4
  8 4 1 8 8 7 6 4
  9 4 9 1 6 7 8 6
  6 8 4 6 7 4 1 7
  9 8 1 8 9 9 4 9
共 10 種
時間到！共 70 次交換：成功 66、失敗 4（成功率 94%），等待穩定超時 12 次。
